# Solution 03: Fine-Tuning GLiClass

In [1]:
import json, os, random, warnings, tempfile
warnings.filterwarnings('ignore')
import torch
import numpy as np
from sklearn.metrics import accuracy_score
from gliclass import GLiClassModel, ZeroShotClassificationPipeline
from gliclass.data_processing import GLiClassDataset, DataCollatorWithPadding, AugmentationConfig
from gliclass.training import TrainingArguments, Trainer
from transformers import AutoTokenizer

fixtures = os.path.normpath(os.path.join(os.getcwd(), "..", "fixtures", "input"))
with open(os.path.join(fixtures, "classification_training_data.json")) as f:
    data = json.load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LABELS = ["technology", "finance", "sports", "science", "politics"]

random.seed(42)
random.shuffle(data)
split = int(len(data) * 0.8)
train_data, eval_data = data[:split], data[split:]
print(f"✓ Loaded {len(data)} examples — Train: {len(train_data)}, Eval: {len(eval_data)}")

✓ Loaded 50 examples — Train: 40, Eval: 10


## Part 1: Zero-Shot Baseline

In [2]:
eval_texts = [d['text'] for d in eval_data]
eval_true = [d['true_labels'][0] for d in eval_data]

model = GLiClassModel.from_pretrained("knowledgator/gliclass-edge-v3.0")
tokenizer = AutoTokenizer.from_pretrained("knowledgator/gliclass-edge-v3.0", add_prefix_space=True)
pipeline = ZeroShotClassificationPipeline(
    model, tokenizer, classification_type='single-label', device=device
)

baseline_results = pipeline(eval_texts, LABELS, batch_size=4)
baseline_preds = [r[0]['label'] for r in baseline_results]
acc_before = sum(p == t for p, t in zip(baseline_preds, eval_true)) / len(eval_true)

assert isinstance(acc_before, float)
assert 0.0 <= acc_before <= 1.0
print(f"✓ Zero-shot accuracy: {acc_before:.0%} ({int(acc_before * len(eval_data))}/{len(eval_data)})")

  0%|          | 0/3 [00:00<?, ?it/s]

 33%|███▎      | 1/3 [00:02<00:05,  2.68s/it]

100%|██████████| 3/3 [00:02<00:00,  1.11it/s]

✓ Zero-shot accuracy: 80% (8/10)


## Part 2: Fine-Tune the Model

In [3]:
output_dir = tempfile.mkdtemp(prefix="gliclass_ft_")

def compute_metrics(p):
    preds, labels = p
    preds_flat = (preds.reshape(-1) > 0.5).astype(int)
    labels_flat = (labels.reshape(-1) > 0.5).astype(int)
    return {"accuracy": float(accuracy_score(labels_flat, preds_flat))}

augment_config = AugmentationConfig(enabled=False)

train_dataset = GLiClassDataset(
    train_data, tokenizer, augment_config,
    max_length=256,
    problem_type='multi_label_classification',
    architecture_type='uni-encoder',
    prompt_first=True
)
eval_dataset = GLiClassDataset(
    eval_data, tokenizer, augment_config,
    max_length=256,
    problem_type='multi_label_classification',
    architecture_type='uni-encoder',
    prompt_first=True
)
data_collator = DataCollatorWithPadding(device=device)

training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    max_steps=150,
    learning_rate=3e-5,
    others_lr=1e-4,
    warmup_ratio=0.1,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=75,
    save_steps=1000,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print(f"Fine-tuning for {training_args.max_steps} steps...")
trainer.train()

assert hasattr(trainer, 'state')
assert trainer.state.global_step >= 100
print(f"✓ Trained for {trainer.state.global_step} steps")

Total labels:  5
Total labels:  5
Fine-tuning for 150 steps...


Step,Training Loss,Validation Loss,Accuracy
75,0.153000,0.049050,0.940000
150,0.000000,0.004817,0.980000


✓ Trained for 150 steps


## Part 3: Compare Before vs After

In [4]:
ft_results = pipeline(eval_texts, LABELS, batch_size=4)
ft_preds = [r[0]['label'] for r in ft_results]
acc_after = sum(p == t for p, t in zip(ft_preds, eval_true)) / len(eval_true)

assert isinstance(acc_after, float)
assert 0.0 <= acc_after <= 1.0
assert acc_after >= 0.70, f"Fine-tuned accuracy {acc_after:.0%} < 70%"
improvement = acc_after - acc_before
assert improvement >= 0.10 or acc_after >= 0.90

print(f"✓ Zero-shot: {acc_before:.0%} → Fine-tuned: {acc_after:.0%} (Δ={improvement:+.0%})")
print()
for pred, true, text in zip(ft_preds, eval_true, eval_texts):
    mark = "✓" if pred == true else "✗"
    print(f"  {mark} [{pred:12}] true={true:12} | {text[:50]}...")

  0%|          | 0/3 [00:00<?, ?it/s]

100%|██████████| 3/3 [00:00<00:00, 75.26it/s]

✓ Zero-shot: 80% → Fine-tuned: 100% (Δ=+20%)

  ✓ [technology  ] true=technology   | A new open-source database engine written in Rust ...
  ✓ [science     ] true=science      | A study published in Nature demonstrated that octo...
  ✓ [technology  ] true=technology   | Tesla released a major over-the-air update adding ...
  ✓ [technology  ] true=technology   | Quantum computing company IonQ demonstrated a 40-q...
  ✓ [finance     ] true=finance      | Crude oil prices dropped to a six-month low amid r...
  ✓ [finance     ] true=finance      | Hedge fund Bridgewater Associates reported its All...
  ✓ [finance     ] true=finance      | Venture capital funding for AI startups reached $2...
  ✓ [technology  ] true=technology   | Google DeepMind released a new large language mode...
  ✓ [technology  ] true=technology   | A new programming language designed for AI systems...
  ✓ [politics    ] true=politics     | The US Senate passed a bipartisan infrastructure b...
